# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

Welcome to the **LeRobot SmolVLA training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `SmolVLA` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `SmolVLA` policy for 20,000 steps typically takes **about 5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer!

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install conda
This cell uses `condacolab` to bootstrap a full Conda environment inside Google Colab.


In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg (version 7.1.1), and installs the package in editable mode.


In [2]:
!conda install python=3.12 -y
!python --version

Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - python=3.12


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    brotli-python-1.2.0        |  py312hdb49522_1         360 KB  conda-forge
    c-ares-1.34.6              |       hb03c661_0         203 KB  conda-forge
    ca-certificates-2026.2.25  |       hbd8a1cb_0         144 KB  conda-forge
    certifi-2026.2.25          |     pyhd8ed1ab_0         148 KB  conda-forge
    cffi-2.0.0                 |  py312h460c074_1         289 KB  conda-forge
    conda-26.1.1               |  py312h7900ff3_0         1.2 MB  conda-forge
    conda-libmamba-solver-25.11.0|     pyhd8ed1ab_1          56 KB  conda-forge
    cpp-expected-1.3.1         |       h171cf75_0          24 KB  conda-forge
    fmt-12.1.0                 |       hff5e90

In [3]:
!rm -rf lerobot
!git clone https://github.com/mdas64/lerobot.git
!conda install ffmpeg=7.1.1 -c conda-forge
!cd lerobot && pip install -e .

Cloning into 'lerobot'...
remote: Enumerating objects: 16566, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (147/147), done.
remote: Total 16566 (delta 56), reused 17 (delta 17), pack-reused 16402 (from 3)
Receiving objects: 100% (16566/16566), 113.06 MiB | 11.59 MiB/s, done.
Resolving deltas: 100% (8673/8673), done.
Filtering content: 100% (45/45), 69.03 MiB | 3.62 MiB/s, done.
Channels:
 - conda-forge
Platform: linux-64
Solving environment: - \ | / done

## Package Plan ##

  environment location: /usr/local

  added / updated specs:
    - ffmpeg=7.1.1


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    alsa-lib-1.2.15.3          |       hb03c661_0         571 KB  conda-forge
    aom-3.9.1                  |       hac33072_0         2.6 MB  conda-forge
    attr-2.5.2                 |       hb03c661_1          31 KB  conda-forge
    cairo

## Weights & Biases login
This cell logs you into Weights & Biases (wandb) to enable experiment tracking and logging.

In [5]:
!wandb login

wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter: 
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: milpdio (milpdio-georgia-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Install SmolVLA dependencies

In [15]:
!cd lerobot && pip install -e ".[smolvla]"

Obtaining file:///content/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.5.1-0.editable-py3-none-any.whl size=12938 sha256=54ad64f820cec3c6280b706f0e231fb2eb00112812c628b7311091fb34d5e6d9
  Stored in directory: /tmp/pip-ephem-wheel-cache-r7s38wcg/wheels/09/b4/fe/75732b1d640db96ba1f856f2b7328b232a03b696a39cb59686
Successfully built lerobot
  Attempting uninstall: lerobot
    Found existing installation: lerobot 0.5.1
    Uninstalling lerobot-0.5.1:
      Successfully uninstalled lerobot-0.5.1


In [23]:
!pip uninstall libero -y

# Clone the repo directly — most reliable method
!git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git /content/LIBERO

Found existing installation: libero 0.1.0
Uninstalling libero-0.1.0:
  Successfully uninstalled libero-0.1.0
Cloning into '/content/LIBERO'...
remote: Enumerating objects: 1788, done.
remote: Counting objects: 100% (582/582), done.
remote: Compressing objects: 100% (210/210), done.
remote: Total 1788 (delta 380), reused 372 (delta 372), pack-reused 1206 (from 1)
Receiving objects: 100% (1788/1788), 315.98 MiB | 16.02 MiB/s, done.
Resolving deltas: 100% (755/755), done.
Updating files: 100% (1116/1116), done.


In [24]:
import sys
import os

# Add to path FIRST before installing
sys.path.insert(0, '/content/LIBERO')

# Install in editable mode so the libero/ directory is directly importable
!pip install -e /content/LIBERO --no-deps -q

  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of libero==0.1.0 from file:///content/LIBERO (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457


In [25]:
!find /content/LIBERO -name "benchmark.py" 2>/dev/null
!ls /content/LIBERO/libero/

configs  libero  lifelong


In [26]:
!ls /content/LIBERO/libero/libero/
!find /content/LIBERO -name "__init__.py" | head -20
!find /content/LIBERO -name "benchmark.py"

assets	bddl_files  benchmark  envs  init_files  __init__.py  utils
/content/LIBERO/libero/lifelong/models/__init__.py
/content/LIBERO/libero/lifelong/__init__.py
/content/LIBERO/libero/lifelong/algos/__init__.py
/content/LIBERO/libero/libero/benchmark/__init__.py
/content/LIBERO/libero/libero/envs/problems/__init__.py
/content/LIBERO/libero/libero/envs/regions/__init__.py
/content/LIBERO/libero/libero/envs/objects/__init__.py
/content/LIBERO/libero/libero/envs/__init__.py
/content/LIBERO/libero/libero/envs/robots/__init__.py
/content/LIBERO/libero/libero/envs/object_states/__init__.py
/content/LIBERO/libero/libero/envs/arenas/__init__.py
/content/LIBERO/libero/libero/envs/predicates/__init__.py
/content/LIBERO/libero/libero/utils/__init__.py
/content/LIBERO/libero/libero/__init__.py
/content/LIBERO/libero/configs/__init__.py


In [27]:
# 1. Create the missing __init__.py
!touch /content/LIBERO/libero/__init__.py
!echo "✅ Created:" && ls -la /content/LIBERO/libero/__init__.py

# 2. Ensure correct path
import sys
if '/content/LIBERO' not in sys.path:
    sys.path.insert(0, '/content/LIBERO')

# 3. Set rendering env vars BEFORE importing
import os
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

# 4. Test import
from libero.libero import benchmark, get_libero_path
print("✅ LIBERO imported successfully")

✅ Created:
-rw-r--r-- 1 root root 0 Mar 19 19:04 /content/LIBERO/libero/__init__.py
Do you want to specify a custom path for the dataset folder? (Y/N): y
Enter the path where you want to store the datasets: /content/libero_data
The full path of the custom storage path you entered is:
/content/libero_data/datasets
Do you want to continue? (Y/N)
y
Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /content/LIBERO/libero/libero
bddl_files: /content/LIBERO/libero/libero/./bddl_files
init_states: /content/LIBERO/libero/libero/./init_files
datasets: /content/libero_data/datasets
assets: /content/LIBERO/libero/libero/./assets
✅ LIBERO imported successfully


In [29]:
import os

# These persist into subprocesses
os.environ["PYTHONPATH"] = "/content/LIBERO:" + os.environ.get("PYTHONPATH", "")
os.environ["MUJOCO_GL"] = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

In [32]:
!pip install mujoco numba scipy opencv-python pytest qpsolvers[quadprog] mink==0.0.5

  Using cached numba-0.64.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.9 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached pytest-9.0.2-py3-none-any.whl.metadata (7.6 kB)
  Using cached mink-0.0.5-py3-none-any.whl.metadata (5.5 kB)
  Using cached qpsolvers-4.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached llvmlite-0.46.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.0 kB)
INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
  Using cached opencv_python-4.13.0.90-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached opencv_python-4.12.0.88-cp37-abi3-manyl

## Start training SmolVLA with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--batch_size=64`: means the model processes 64 training samples in parallel before doing one gradient update. Reduce this number if you have a GPU with low memory.

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this.

In [33]:
!cd /content/lerobot && PYTHONPATH="/content/LIBERO:$PYTHONPATH" \
  MUJOCO_GL=osmesa \
  PYOPENGL_PLATFORM=osmesa \
  python src/lerobot/scripts/lerobot_train.py \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --env.type=libero \
  --env.task=libero_object \
  --batch_size=4 \
  --steps=100 \
  --output_dir=outputs/train/my_smolvla \
  --job_name=my_smolvla_training \
  --policy.device=cuda \
  --policy.push_to_hub=false

INFO 2026-03-19 19:08:47 ot_train.py:197 {'batch_size': 4,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                            

In [28]:
!cd lerobot && python src/lerobot/scripts/lerobot_train.py \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id=HuggingFaceVLA/libero \
  --env.type=libero \
  --env.task=libero_object \
  --batch_size=4 \
  --steps=100 \
  --output_dir=outputs/train/my_smolvla \
  --job_name=my_smolvla_training \
  --policy.device=cuda \
  --policy.push_to_hub=false

INFO 2026-03-19 19:05:12 ot_train.py:197 {'batch_size': 4,
 'checkpoint_path': None,
 'cudnn_deterministic': False,
 'dataset': {'episodes': None,
             'image_transforms': {'enable': False,
                                  'max_num_transforms': 3,
                                  'random_order': False,
                                  'tfs': {'affine': {'kwargs': {'degrees': [-5.0,
                                                                            5.0],
                                                                'translate': [0.05,
                                                                              0.05]},
                                                     'type': 'RandomAffine',
                                                     'weight': 1.0},
                                          'brightness': {'kwargs': {'brightness': [0.8,
                                                                                   1.2]},
                            

## Login into Hugging Face Hub
Now after training is done login into the Hugging Face hub and upload the last checkpoint

In [ ]:
!huggingface-cli login

In [ ]:
!huggingface-cli upload $milpdio/my_smolvla \
  /content/lerobot/outputs/train/my_smolvla/checkpoints/last/pretrained_model